In [1]:
import os
import glob
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from models.db_models import TrainingDataGenMetaData, StoryverseMetaData, TrainingIOPair
from llm_util import OpenAiLLMProvider, LLMProvider
from db_utils import DatabaseManager

load_dotenv()

True

In [2]:
def extract_response_text(response) -> str:
    """Extract text from LLM response output"""
    if not response or not response.output:
        raise ValueError("Invalid response from LLM provider")
    
    for output in response.output:
        if output.type == "message" and output.content and len(output.content) > 0:
            return output.content[0].text
    
    return ""

In [3]:
def generateTrainingIOPair(
    first_draft_file_path: str,
    training_meta_data: TrainingDataGenMetaData,
    storyverse_meta_data: StoryverseMetaData,
    llm_provider: LLMProvider,
    db_manager: DatabaseManager,
    story_verse: str = "SHERLOCK"
) -> dict:
    """Generate training IO pairs for all pipeline steps from a first draft story"""
    
    print(f"\nProcessing file: {first_draft_file_path}")
    
    # Read the first draft
    with open(first_draft_file_path, 'r', encoding='utf-8') as f:
        first_draft = f.read()
    
    # Store the generated outputs for reference in later steps
    generated_outputs = {}
    created_pairs = []
    
    # STEP 1: CHARACTER_DATA_GEN
    print("  - Generating CHARACTER_DATA_GEN pair...")
    char_pair = TrainingIOPair(
        storyVerse=story_verse,
        pipelineStepName="CHARACTER_DATA_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=datetime.utcnow().isoformat()
    )
    
    # Generate model output using reverse prompt
    prompt = f"{training_meta_data.characterGenearationReversePrompt}\n\n{first_draft}"
    char_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    char_pair.modelOutput = extract_response_text(response)
    generated_outputs["CHARACTER_DATA_GEN"] = char_pair.modelOutput
    
    # Generate input prompt for fine-tuning
    prompt = f"{training_meta_data.characterInputPomptGenrationPrompt}\n\n{char_pair.modelOutput}"
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    char_pair.inputPromptForFineTuning = extract_response_text(response)
    
    # Save to DB
    char_id = db_manager.create_training_io_pair(char_pair)
    created_pairs.append(("CHARACTER_DATA_GEN", char_id))
    print(f"    ✓ Saved CHARACTER_DATA_GEN pair with ID: {char_id}")
    
    # STEP 2: PLOT_GEN
    print("  - Generating PLOT_GEN pair...")
    plot_pair = TrainingIOPair(
        storyVerse=story_verse,
        pipelineStepName="PLOT_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=datetime.utcnow().isoformat()
    )
    
    # Generate model output
    prompt = f"{training_meta_data.plotGenerationReversePrompt}\n\n{first_draft}"
    plot_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    plot_pair.modelOutput = extract_response_text(response)
    generated_outputs["PLOT_GEN"] = plot_pair.modelOutput
    
    # Generate input prompt with CHARACTER_DATA replacement
    prompt_template = training_meta_data.plotGenerationPomptGenrationPrompt
    prompt_with_char_data = prompt_template.replace(
        "{{CHARACTER_DATA}}",
        generated_outputs["CHARACTER_DATA_GEN"]
    )
    prompt = f"{prompt_with_char_data}\n\n{plot_pair.modelOutput}"
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    plot_pair.inputPromptForFineTuning = extract_response_text(response)
    
    # Save to DB
    plot_id = db_manager.create_training_io_pair(plot_pair)
    created_pairs.append(("PLOT_GEN", plot_id))
    print(f"    ✓ Saved PLOT_GEN pair with ID: {plot_id}")
    
    # STEP 3: STORY_CHAIN_GEN
    print("  - Generating STORY_CHAIN_GEN pair...")
    chain_pair = TrainingIOPair(
        storyVerse=story_verse,
        pipelineStepName="STORY_CHAIN_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=datetime.utcnow().isoformat()
    )
    
    # Generate model output
    prompt = f"{training_meta_data.storyChainGenerationReversePrompt}\n\n{first_draft}"
    chain_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    chain_pair.modelOutput = extract_response_text(response)
    generated_outputs["STORY_CHAIN_GEN"] = chain_pair.modelOutput
    
    # Generate input prompt with CHARACTER_DATA and PLOT replacements
    prompt_template = storyverse_meta_data.storyChainGenerationPromptTemplate.promptTemplate
    chain_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{PLOT}}", generated_outputs["PLOT_GEN"]
    )
    
    # Save to DB
    chain_id = db_manager.create_training_io_pair(chain_pair)
    created_pairs.append(("STORY_CHAIN_GEN", chain_id))
    print(f"    ✓ Saved STORY_CHAIN_GEN pair with ID: {chain_id}")
    
    # STEP 4: STORY_SUMMARY_GEN
    print("  - Generating STORY_SUMMARY_GEN pair...")
    summary_pair = TrainingIOPair(
        storyVerse=story_verse,
        pipelineStepName="STORY_SUMMARY_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=datetime.utcnow().isoformat()
    )
    
    # Generate model output
    prompt = f"{training_meta_data.storySummaryGenerationReversePrompt}\n\n{first_draft}"
    summary_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-5-2025-08-07",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    summary_pair.modelOutput = extract_response_text(response)
    generated_outputs["STORY_SUMMARY_GEN"] = summary_pair.modelOutput
    
    # Generate input prompt with CHARACTER_DATA, PLOT, and STORY_CHAIN replacements
    prompt_template = storyverse_meta_data.storySummaryGenerationPromptTemplate.promptTemplate
    summary_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{PLOT}}", generated_outputs["PLOT_GEN"]
    ).replace(
        "{{STORY_CHAIN}}", generated_outputs["STORY_CHAIN_GEN"]
    )
    
    # Save to DB
    summary_id = db_manager.create_training_io_pair(summary_pair)
    created_pairs.append(("STORY_SUMMARY_GEN", summary_id))
    print(f"    ✓ Saved STORY_SUMMARY_GEN pair with ID: {summary_id}")
    
    # STEP 5: FIRST_DRAFT_GEN
    print("  - Generating FIRST_DRAFT_GEN pair...")
    draft_pair = TrainingIOPair(
        storyVerse=story_verse,
        pipelineStepName="FIRST_DRAFT_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=datetime.utcnow().isoformat()
    )
    
    # The first draft itself is the model output - no need to reverse-engineer it
    draft_pair.modelOutput = first_draft
    generated_outputs["FIRST_DRAFT_GEN"] = draft_pair.modelOutput
    
    # For FIRST_DRAFT_GEN, reverseOutputPrompt is not applicable since we don't reverse-engineer
    # We can store the template for reference or leave it empty
    draft_pair.reverseOutputPrompt = training_meta_data.fistDraftGenerationReversePrompt
    
    # Generate input prompt with CHARACTER_DATA and STORY_SUMMARY replacements
    prompt_template = storyverse_meta_data.fistDraftGenerationPromptTemplate.promptTemplate
    draft_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{STORY_SUMMARY}}", generated_outputs["STORY_SUMMARY_GEN"]
    )
    
    # Save to DB
    draft_id = db_manager.create_training_io_pair(draft_pair)
    created_pairs.append(("FIRST_DRAFT_GEN", draft_id))
    print(f"    ✓ Saved FIRST_DRAFT_GEN pair with ID: {draft_id}")
    
    return {
        "file": first_draft_file_path,
        "pairs": created_pairs
    }

In [4]:
if __name__ == "__main__":
    # Initialize providers
    llm_provider = OpenAiLLMProvider()
    db_manager = DatabaseManager()
    
    # Get story verse from environment
    story_verse = os.getenv('STORY_VERSE', 'SHERLOCK')
    
    print(f"=== Training Data Generation Pipeline for {story_verse} ===")
    print("\n1. Fetching metadata from database...")
    
    # Fetch TrainingDataGenMetaData
    training_meta_data = db_manager.get_training_data_gen_meta_data(story_verse)
    if not training_meta_data:
        raise ValueError(f"No TrainingDataGenMetaData found for storyVerse: {story_verse}")
    print(f"   ✓ Found TrainingDataGenMetaData for {story_verse}")
    
    # Fetch StoryverseMetaData
    storyverse_meta_data = db_manager.get_meta_data(story_verse)
    if not storyverse_meta_data:
        raise ValueError(f"No StoryverseMetaData found for storyVerse: {story_verse}")
    print(f"   ✓ Found StoryverseMetaData for {story_verse}")
    
    # Get all text files from first-drafts folder
    first_drafts_dir = f"training-data/{story_verse}/first-drafts"
    text_files = glob.glob(os.path.join(first_drafts_dir, "*.txt"))
    
    if not text_files:
        print(f"\n⚠ No text files found in {first_drafts_dir}")
        print("Please add first draft stories to this directory and run again.")
    else:
        print(f"\n2. Found {len(text_files)} first draft files to process")
        for f in text_files:
            print(f"   - {os.path.basename(f)}")
        
        print(f"\n3. Processing files with thread pool (max 3 concurrent threads)...")
        
        results = []
        errors = []
        
        # Use ThreadPoolExecutor for parallel processing
        with ThreadPoolExecutor(max_workers=3) as executor:
            # Submit all tasks
            future_to_file = {
                executor.submit(
                    generateTrainingIOPair,
                    file_path,
                    training_meta_data,
                    storyverse_meta_data,
                    llm_provider,
                    db_manager,
                    story_verse
                ): file_path for file_path in text_files
            }
            
            # Collect results as they complete
            for future in as_completed(future_to_file):
                file_path = future_to_file[future]
                try:
                    result = future.result()
                    results.append(result)
                except Exception as exc:
                    error_msg = f"Error processing {file_path}: {exc}"
                    print(f"\n❌ {error_msg}")
                    errors.append(error_msg)
        
        # Print summary
        print(f"\n{'='*60}")
        print(f"Pipeline Complete!")
        print(f"{'='*60}")
        print(f"Successfully processed: {len(results)}/{len(text_files)} files")
        print(f"Total training pairs created: {len(results) * 5}")
        
        if errors:
            print(f"\n⚠ Errors encountered: {len(errors)}")
            for error in errors:
                print(f"   - {error}")
        
        if results:
            print(f"\n✓ Successfully generated training pairs for:")
            for result in results:
                print(f"   {os.path.basename(result['file'])}:")
                for step_name, pair_id in result['pairs']:
                    print(f"      - {step_name}: {pair_id}")
    
    # Close DB connection
    db_manager.close()
    print("\n✓ Database connection closed")

=== Training Data Generation Pipeline for SHERLOCK ===

1. Fetching metadata from database...
   ✓ Found TrainingDataGenMetaData for SHERLOCK
   ✓ Found StoryverseMetaData for SHERLOCK

2. Found 3 first draft files to process
   - scandal-in-bohemia.txt
   - adventure-of-cardboard-box.txt
   - adventure-of-empty-house.txt

3. Processing files with thread pool (max 3 concurrent threads)...

Processing file: training-data/SHERLOCK/first-drafts/scandal-in-bohemia.txt

Processing file: training-data/SHERLOCK/first-drafts/adventure-of-cardboard-box.txt

Processing file: training-data/SHERLOCK/first-drafts/adventure-of-empty-house.txt
  - Generating CHARACTER_DATA_GEN pair...
  - Generating CHARACTER_DATA_GEN pair...
  - Generating CHARACTER_DATA_GEN pair...


/var/folders/2l/bl9kggyn1b14p4wk0brvxdh00000gn/T/ipykernel_20385/4151654519.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  createdAt=datetime.utcnow().isoformat()


    ✓ Saved CHARACTER_DATA_GEN pair with ID: 68eac974eef30fce33b7aedd
  - Generating PLOT_GEN pair...


/var/folders/2l/bl9kggyn1b14p4wk0brvxdh00000gn/T/ipykernel_20385/4151654519.py:69: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  createdAt=datetime.utcnow().isoformat()


    ✓ Saved CHARACTER_DATA_GEN pair with ID: 68eac996eef30fce33b7aede
  - Generating PLOT_GEN pair...
    ✓ Saved CHARACTER_DATA_GEN pair with ID: 68eac9e9eef30fce33b7aedf
  - Generating PLOT_GEN pair...
    ✓ Saved PLOT_GEN pair with ID: 68eaca23eef30fce33b7aee0
  - Generating STORY_CHAIN_GEN pair...


...

    ✓ Saved PLOT_GEN pair with ID: 68eaca34eef30fce33b7aee1
  - Generating STORY_CHAIN_GEN pair...

❌ Error processing training-data/SHERLOCK/first-drafts/scandal-in-bohemia.txt: OpenAI API error: 500 - {
  "error": {
    "message": "The server had an error processing your request. Sorry about that! You can retry your request, or contact us through our help center at help.openai.com if you keep seeing this error. (Please include the request ID req_cdfffa912d20463587b2b7cfa3012722 in your email.)",
    "type": "server_error",
    "param": null,
    "code": null
  }
}
    ✓ Saved PLOT_GEN pair with ID: 68eaca6aeef30fce33b7aee2
  - Generating STORY_CHAIN_GEN pair...
    ✓ Saved STORY_CHAIN_GEN pair with ID: 68eaca6deef30fce33b7aee3
  - Generating STORY_SUMMARY_GEN pair...


/var/folders/2l/bl9kggyn1b14p4wk0brvxdh00000gn/T/ipykernel_20385/4151654519.py:150: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  createdAt=datetime.utcnow().isoformat()


    ✓ Saved STORY_CHAIN_GEN pair with ID: 68eacad7eef30fce33b7aee4
  - Generating STORY_SUMMARY_GEN pair...
    ✓ Saved STORY_SUMMARY_GEN pair with ID: 68eacb16eef30fce33b7aee5
  - Generating FIRST_DRAFT_GEN pair...
    ✓ Saved FIRST_DRAFT_GEN pair with ID: 68eacb16eef30fce33b7aee6


/var/folders/2l/bl9kggyn1b14p4wk0brvxdh00000gn/T/ipykernel_20385/4151654519.py:189: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  createdAt=datetime.utcnow().isoformat()


    ✓ Saved STORY_SUMMARY_GEN pair with ID: 68eacb69eef30fce33b7aee7
  - Generating FIRST_DRAFT_GEN pair...
    ✓ Saved FIRST_DRAFT_GEN pair with ID: 68eacb69eef30fce33b7aee8

Pipeline Complete!
Successfully processed: 2/3 files
Total training pairs created: 10

⚠ Errors encountered: 1
   - Error processing training-data/SHERLOCK/first-drafts/scandal-in-bohemia.txt: OpenAI API error: 500 - {
  "error": {
    "message": "The server had an error processing your request. Sorry about that! You can retry your request, or contact us through our help center at help.openai.com if you keep seeing this error. (Please include the request ID req_cdfffa912d20463587b2b7cfa3012722 in your email.)",
    "type": "server_error",
    "param": null,
    "code": null
  }
}

✓ Successfully generated training pairs for:
   adventure-of-cardboard-box.txt:
      - CHARACTER_DATA_GEN: 68eac996eef30fce33b7aede
      - PLOT_GEN: 68eaca23eef30fce33b7aee0
      - STORY_CHAIN_GEN: 68eaca6deef30fce33b7aee3
      - ST